In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss

In [2]:


matches = pd.read_csv("../data/processed/features_v4.csv")

season_order = {
    "21-22": 0,
    "22-23": 1,
    "23-24": 2,
    "24-25": 3,
    "25-26": 4
}

matches["SeasonOrder"] = matches["Season"].map(season_order)

C:\Users\harry\AppData\Local\Temp\ipykernel_27120\339640967.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches["SeasonOrder"] = matches["Season"].map(season_order)


In [3]:
feature_sets = {
    "Elo only": [
        "EloDiff"
    ],

    "Recent form": [
        "PPGDiff",
        "GoalAgainstDiffLast5",
        "ShotOTDiffLast5"
    ],

    "Long-term": [
        "PPGDiff",
        "GDPerGameDiff",
        "GoalsAgainstPerGameDiff"
    ],

    "Elo + Recent form": [
        "EloDiff",
        "PPGDiff",
        "GoalAgainstDiffLast5",
        "ShotOTDiffLast5"
    ],

    "Elo + Long-term": [
        "EloDiff",
        "PPGDiff",
        "GDPerGameDiff",
        "GoalsAgainstPerGameDiff"
    ],

    "All features": [
        "PPGDiff",
        "GoalAgainstDiffLast5",
        "ShotOTDiffLast5",
        "GDPerGameDiff",
        "GoalsAgainstPerGameDiff",
        "EloDiff"
    ]
}

In [4]:
seasons = [
    "22-23",
    "23-24",
    "24-25",
    "25-26"
]

results = []

for feature_set_name, features in feature_sets.items():

    for test_season in seasons:

        train = matches[
            matches["SeasonOrder"] < season_order[test_season]
        ]

        test = matches[
            matches["Season"] == test_season
        ]

        X_train = train[features]
        y_train = train["FTR"]

        X_test = test[features]
        y_test = test["FTR"]

        model = LogisticRegression(
            max_iter=1000
        )

        model.fit(X_train, y_train)

        preds = model.predict(X_test)
        probs = model.predict_proba(X_test)

        accuracy = accuracy_score(y_test, preds)

        loss = log_loss(
            y_test,
            probs,
            labels=model.classes_
        )

        results.append({
            "FeatureSet": feature_set_name,
            "Season": test_season,
            "Accuracy": accuracy,
            "LogLoss": loss
        })

In [5]:
results_df = pd.DataFrame(results)

results_df

,FeatureSet,Season,Accuracy,LogLoss
0,Elo only,22-23,0.552632,0.999150
1,Elo only,23-24,0.557895,0.940447
2,Elo only,24-25,0.536842,0.997357
3,Elo only,25-26,0.484211,1.032826
4,Recent form,22-23,0.528947,0.999905
5,Recent form,23-24,0.571053,0.953668
6,Recent form,24-25,0.534211,1.005709
7,Recent form,25-26,0.481579,1.050434
8,Long-term,22-23,0.528947,0.998702
9,Long-term,23-24,0.573684,0.964825


In [6]:
season_order = {
    "21-22": 0,
    "22-23": 1,
    "23-24": 2,
    "24-25": 3,
    "25-26": 4
}

matches["SeasonOrder"] = matches["Season"].map(season_order)

In [8]:
summary = (
    results_df
    .groupby("FeatureSet")[["Accuracy", "LogLoss"]]
    .mean()
    .sort_values("LogLoss")
)

summary

,Accuracy,LogLoss
FeatureSet,,
Elo + Recent form,0.540789,0.990375
Elo + Long-term,0.526316,0.990996
All features,0.535526,0.992273
Elo only,0.532895,0.992445
Recent form,0.528947,1.002429
Long-term,0.521711,1.007898


In [9]:
results_df.pivot(
    index="FeatureSet",
    columns="Season",
    values="Accuracy"
)

Season,22-23,23-24,24-25,25-26
FeatureSet,,,,
All features,0.550000,0.571053,0.536842,0.484211
Elo + Long-term,0.547368,0.552632,0.526316,0.478947
Elo + Recent form,0.560526,0.568421,0.544737,0.489474
Elo only,0.552632,0.557895,0.536842,0.484211
Long-term,0.528947,0.573684,0.502632,0.481579
Recent form,0.528947,0.571053,0.534211,0.481579


In [10]:
results_df.pivot(
    index="FeatureSet",
    columns="Season",
    values="LogLoss"
)

Season,22-23,23-24,24-25,25-26
FeatureSet,,,,
All features,1.002843,0.935193,0.995080,1.035975
Elo + Long-term,0.992839,0.942597,0.992832,1.035716
Elo + Recent form,0.996996,0.935206,0.994777,1.034521
Elo only,0.999150,0.940447,0.997357,1.032826
Long-term,0.998702,0.964825,1.011234,1.056833
Recent form,0.999905,0.953668,1.005709,1.050434
